# 사내 업무 가이드 RAG

`data/가상Tech_업무가이드.pdf`를 근거로 질문에 답하는 RAG 챗봇을 단계별로 구성한다.

흐름: **Loader → TextSplitter → Embedding → VectorStore → Retriever → Prompt → LLM → OutputParser → Chain**

## 1. Loader
PDF를 페이지 단위 `Document`로 읽어온다.

In [35]:
from pathlib import Path
from langchain_community.document_loaders import PyPDFLoader

PDF_PATH = Path("data") / "가상Tech_업무가이드.pdf"

loader = PyPDFLoader(str(PDF_PATH))
pages = loader.load()

print(f"페이지 수: {len(pages)}")
print(f"메타데이터: {pages[0].metadata}")
print(pages[0].page_content[:300])

페이지 수: 9
메타데이터: {'producer': 'ReportLab PDF Library - (opensource)', 'creator': '(unspecified)', 'creationdate': '2026-09-26T11:41:50+09:00', 'author': '가상Tech People Operations', 'keywords': '', 'moddate': '2026-09-26T11:41:50+09:00', 'subject': 'RAG 실습용 가상 사내 업무 가이드', 'title': '가상Tech 업무 가이드', 'trapped': '/False', 'source': 'data\\가상Tech_업무가이드.pdf', 'total_pages': 9, 'page': 0, 'page_label': '1'}
VIRTUALTECH · INTERNAL HANDBOOK
가상Tech
업무 가이드
함께 일하는 방식부터 보안, 개발, 장애 대응, 복지까지
구성원이 매일 참고하는 실무 기준
 문서 버전
 시행일
 문서 등급
 문의
v1.0
2026-09-01
사내 공개
People Operations
이 문서의 목적
가상Tech 구성원이 반복해서 찾는 사내 정책과 표준 절차를 한 문서에 정리했습니다.
실제 회사가 아닌 RAG 검색·질의응답 실습용 가상 문서입니다.


## 2. TextSplitter
문서에 표(휴가·비용 등)가 많아서 청크를 너무 작게 자르면 표가 끊긴다. 청크를 넉넉하게 잡고, 문단 → 줄 → 문장 순으로 자르도록 한다.

In [36]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

splitter = RecursiveCharacterTextSplitter(
    chunk_size=800,
    chunk_overlap=100,
    separators=["\n\n", "\n", ". ", " ", ""],
)

chunks = splitter.split_documents(pages)

print(f"청크 수: {len(chunks)}")
print(f"청크 길이(최소/최대): {min(len(c.page_content) for c in chunks)} / {max(len(c.page_content) for c in chunks)}")

청크 수: 15
청크 길이(최소/최대): 153 / 793


In [37]:
for c in chunks[:3]:
    print(f"[page {c.metadata['page'] + 1}] {len(c.page_content)}자")
    print(c.page_content)
    print("-" * 40)

[page 1] 252자
VIRTUALTECH · INTERNAL HANDBOOK
가상Tech
업무 가이드
함께 일하는 방식부터 보안, 개발, 장애 대응, 복지까지
구성원이 매일 참고하는 실무 기준
 문서 버전
 시행일
 문서 등급
 문의
v1.0
2026-09-01
사내 공개
People Operations
이 문서의 목적
가상Tech 구성원이 반복해서 찾는 사내 정책과 표준 절차를 한 문서에 정리했습니다.
실제 회사가 아닌 RAG 검색·질의응답 실습용 가상 문서입니다.
----------------------------------------
[page 2] 768자
가상Tech 업무 가이드
사내용 · v1.0
문서 책임: People Operations
02
CHAPTER 01
회사와 업무 원칙
빠르게 만들되, 고객 신뢰와 동료의 집중 시간을 지킵니다.
 
1.1 회사 개요
가상Tech는 기업용 협업 자동화 SaaS인 ‘FlowPilot’을 개발·운영하는 가상 기업입니다. 본사는 서울 성수동에 있으며
제품개발, 고객성공, 영업, 경영지원 조직으로 구성됩니다. 표준 근무시간은 평일 10:00-19:00이고, 핵심 협업시간은
11:00-16:00입니다.
1.2 우리가 일하는 다섯 가지 원칙
 • 고객 영향부터 판단한다: 기능보다 고객 데이터와 서비스 연속성을 우선한다.
 • 기록이 회의를 이긴다: 결정, 근거, 담당자, 기한을 문서에 남긴다.
 • 작게 배포하고 빠르게 검증한다: 변경 범위를 줄이고 되돌릴 수 있게 만든다.
 • 문제는 사람보다 시스템에서 찾는다: 회고는 비난 없이 재발 방지에 집중한다.
 • 기본 공개, 필요한 경우 제한: 업무 정보는 사내 공개가 원칙이며 개인정보와 고객정보만 최소 권한으로 제한한다.
 
1.3 근무 방식
 항목
 기준
 세부 규칙
출근
유연근무
08:00-11:00 사이 출근, 일 8시간 근무
재택
주 2회
매주 월요일까지 캘린더에 ‘WFH’ 표시
회의
25분/50분
안건 없는 회의는 거절 가능, 회의록은 24시간 내 등록

## 3. Embedding
환경변수(.env)를 읽고 OpenAI 임베딩 모델을 준비한다.

In [38]:
from dotenv import load_dotenv
from langchain_openai import OpenAIEmbeddings

load_dotenv()

embedding = OpenAIEmbeddings(model="text-embedding-3-small")

vec = embedding.embed_query("연차 신청 기한")
print(f"벡터 차원: {len(vec)}")

벡터 차원: 1536


## 4. VectorStore
FAISS에 청크 임베딩을 저장한다. 다시 실행할 때 임베딩 비용을 아끼려고 로컬에 저장/재사용한다.

In [39]:
from langchain_community.vectorstores import FAISS

INDEX_DIR = Path("data") / "faiss_index"

if INDEX_DIR.exists():
    vectorstore = FAISS.load_local(
        str(INDEX_DIR), embedding, allow_dangerous_deserialization=True
    )
    print("저장된 인덱스를 불러왔습니다.")
else:
    vectorstore = FAISS.from_documents(chunks, embedding)
    vectorstore.save_local(str(INDEX_DIR))
    print("새 인덱스를 만들고 저장했습니다.")

print(f"저장된 벡터 수: {vectorstore.index.ntotal}")

새 인덱스를 만들고 저장했습니다.
저장된 벡터 수: 15


## 5. Retriever
질문과 가장 가까운 청크 k개를 가져온다.

In [40]:
retriever = vectorstore.as_retriever(search_type="similarity", search_kwargs={"k": 5})

def show_search(query):
    for rank, d in enumerate(retriever.invoke(query), 1):
        print(f"[{rank}] page {d.metadata['page'] + 1}: {d.page_content[:80]!r}")

show_search("연차는 며칠 전에 신청해야 하나요?")

[1] page 2: '집중시간\n14:00-16:00\n화·목은 사내 정기회의 생성 금지\n응답 기대\n업무시간 4시간\n긴급은 Slack #help-urgent와 전화 병행'
[2] page 6: '• 금요일 15:00 이후와 공휴일 전날에는 긴급 변경 외 운영 배포를 금지합니다.\n \n운영 DB 접근\n기본 차단입니다. Jira ‘DB-ACC'
[3] page 3: '신청합니다.\n• 퇴사·휴직 시 기기는 마지막 근무일 17:00까지 IT 자산 담당자에게 반납합니다.\n \n피싱 신고\n의심 메일의 링크나 첨부파일을'
[4] page 9: '가상Tech 업무 가이드\n사내용 · v1.0\n문서 책임: People Operations\n09\nCHAPTER 08\nFAQ와 연락처\n헷갈리는 상황'
[5] page 8: '가상Tech 업무 가이드\n사내용 · v1.0\n문서 책임: People Operations\n08\nCHAPTER 07\n휴가·비용·복지\n예측 가능한 '


## 6. Prompt
문서 근거로만 답하고, 근거가 없으면 모른다고 답하게 한다. 출처 페이지도 함께 답하도록 요청한다.

In [41]:
from langchain_core.prompts import ChatPromptTemplate

SYSTEM = (
    "너는 가상Tech 직원을 돕는 사내 규정 안내 도우미다.\n"
    "반드시 아래 [참고 문서]에 있는 내용만 사용해 한국어로 답한다.\n"
    "- 문서에 근거가 없으면 추측하지 말고 '문서에서 확인되지 않습니다.'라고 답한다.\n"
    "- 질문이 여러 개면 항목별로 나눠 답한다.\n"
    "- 서로 다른 규정(예: 신청 시점 vs 정산 기한)을 혼동하지 않는다.\n"
    "- 답변 끝에 근거가 된 페이지를 (p.N) 형식으로 적는다."
)

prompt = ChatPromptTemplate.from_messages([
    ("system", SYSTEM),
    ("human", "[참고 문서]\n{context}\n\n[질문]\n{question}"),
])

## 7. LLM

In [42]:
from langchain_openai import ChatOpenAI

llm = ChatOpenAI(model="gpt-4o-mini", temperature=0)

## 8. OutputParser
`AIMessage`에서 텍스트만 꺼낸다.

In [43]:
from langchain_core.output_parsers import StrOutputParser

parser = StrOutputParser()

## 9. Chain
검색 결과를 `[p.N] 본문` 형태로 이어 붙여 프롬프트에 넣고, LLM → 파서로 연결한다.

In [44]:
from langchain_core.runnables import RunnableLambda, RunnablePassthrough

def format_docs(docs):
    return "\n\n".join(f"[p.{d.metadata['page'] + 1}]\n{d.page_content}" for d in docs)

chain = (
    {"context": retriever | RunnableLambda(format_docs), "question": RunnablePassthrough()}
    | prompt
    | llm
    | parser
)

print(chain.invoke("병가는 언제 신청해야 하나요?"))

병가는 가능한 즉시 신청해야 하며, 3일 초과 시 진료 확인서를 제출해야 합니다. (p.8)


## 10. 질문 10개

In [45]:
questions = [
    "연차를 사용하려면 최소 며칠 전에 신청해야 하나요?",
    "서울 출장 시 숙박비는 최대 얼마까지 지원되나요?",
    "출장을 다녀온 후 비용 정산은 언제까지 완료해야 하나요?",
    "회사 노트북을 분실했을 경우 어떤 절차로 신고해야 하나요?",
    "재택근무를 신청하려면 어떤 조건을 충족해야 하나요?",
    "직원이 업무 관련 교육을 받을 경우 회사에서 지원하는 비용 기준은 어떻게 되나요?",
    "출장 중 택시비를 회사 경비로 처리할 수 있는 경우는 언제인가요?",
    "개인 사정으로 출근 시간이 늦어질 경우 어떤 방식으로 보고해야 하나요?",
    "업무 중 시스템 장애가 발생했을 때 가장 먼저 해야 하는 조치는 무엇인가요?",
    "제주도로 2박 3일 출장을 가는 경우 숙박비 지원 한도와 출장비 정산 기한을 각각 알려주세요.",
]

In [46]:
answers = chain.batch(questions)

for i, (q, a) in enumerate(zip(questions, answers), 1):
    print(f"Q{i}. {q}\nA{i}. {a}\n{'=' * 70}")

Q1. 연차를 사용하려면 최소 며칠 전에 신청해야 하나요?
A1. 문서에서 확인되지 않습니다.
Q2. 서울 출장 시 숙박비는 최대 얼마까지 지원되나요?
A2. 서울 출장 시 숙박비는 1박당 최대 15만원(세금 포함)까지 지원됩니다. (p.8)
Q3. 출장을 다녀온 후 비용 정산은 언제까지 완료해야 하나요?
A3. 출장 비용 정산은 지출일로부터 5영업일 안에 ExpenseHub에 등록해야 합니다. (p.8)
Q4. 회사 노트북을 분실했을 경우 어떤 절차로 신고해야 하나요?
A4. 문서에서 확인되지 않습니다.
Q5. 재택근무를 신청하려면 어떤 조건을 충족해야 하나요?
A5. 문서에서 확인되지 않습니다.
Q6. 직원이 업무 관련 교육을 받을 경우 회사에서 지원하는 비용 기준은 어떻게 되나요?
A6. 직원이 업무 관련 교육을 받을 경우 회사에서 지원하는 비용 기준은 연 120만원입니다. 교육, 도서, 컨퍼런스에 대한 지원이 포함되며, 사전 승인은 LearnHub에서 받아야 합니다. (p.8)
Q7. 출장 중 택시비를 회사 경비로 처리할 수 있는 경우는 언제인가요?
A7. 택시비는 23:00 이후 퇴근, 대중교통 미운행, 고객 일정 등 업무상 필요가 있을 때 실비 지원됩니다. (p.8)
Q8. 개인 사정으로 출근 시간이 늦어질 경우 어떤 방식으로 보고해야 하나요?
A8. 문서에서 확인되지 않습니다.
Q9. 업무 중 시스템 장애가 발생했을 때 가장 먼저 해야 하는 조치는 무엇인가요?
A9. 업무 중 시스템 장애가 발생했을 때 가장 먼저 해야 하는 조치는 감지자가 #help-urgent에 ‘증상, 시작 시각, 영향, 관측 링크’를 게시하고 PagerDuty를 호출하는 것입니다. (p.7)
Q10. 제주도로 2박 3일 출장을 가는 경우 숙박비 지원 한도와 출장비 정산 기한을 각각 알려주세요.
A10. 1. 숙박비 지원 한도: 1박 15만원(세금 포함)입니다.

2. 출장비 정산 기한: 지출일로부터 5영업일 안에 ExpenseHub에 등록해야 합니다.

(p.8)
